# Generalization

## 1. What Is Generalization?

### 1.1 IID Assumption

- **Independent:** Samples are independent of one another.
- **Identically distributed:** Samples are drawn from the same distribution $P$.
- Training, validation, test sets are all assumed to be sampled from $P$.

### 1.2 What Is Generalization?

> Generalization is a model's capability to perform well on unseen samples sampled from the same underlying data distribution.

### 1.3 Generalization Error

Generalization error is the expected loss of the trained model on new samples sampled from $P$.

$$
Err(f) = \mathbb{E}_{x,y \sim p}[\ell(f(x), y)].
$$

However, the true distribution $P$ is unknown. How can we calculate this expectation in practice?

We can use an independent test set to estimate the generalization error:

$$
\widehat{Err}(f) = \mathbb{E}_{x, y \sim \mathcal{D}_{\text{test}}}[\ell(f(x), y)] = \frac{1}{|\mathcal{D}_{\text{test}}|} \sum_{x,y \in \mathcal{D}_{\text{test}}} \ell(f(x), y).
$$

$$
\text{test error} \approx \text{generalization error}
$$

## 2. Why Do We Care about Generalization?

Before answering this question, let's look at a KNN example.

In [ ]:
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

from sklearn import datasets, neighbors

# Make the random splits reproducible
np.random.seed(1234)

In [ ]:
# Load the California housing regression dataset
x, y = datasets.fetch_california_housing(return_X_y=True)

num_instances, num_features = x.shape
print(num_instances, num_features)

# Mean squared error
def loss(y, yh):
    return np.mean((y - yh) ** 2)

In [ ]:
# Use 20% of the data as an unseen test set
n_test = num_instances // 5

# Shuffle the sample indices before splitting
inds = np.random.permutation(num_instances)

x_test, y_test = x[inds[:n_test]], y[inds[:n_test]]
x_train, y_train = x[inds[n_test:]], y[inds[n_test:]]

In [ ]:
K_list = range(1, 100)
err_train, err_test = [], []

for K in K_list:
    model = neighbors.KNeighborsRegressor(n_neighbors=K)
    model.fit(x_train, y_train)

    err_train.append(loss(y_train, model.predict(x_train)))
    err_test.append(loss(y_test, model.predict(x_test)))

> **Pause the video and make a prediction:**
>
> 1. What do you expect the training and test errors to look like when $K=1$?
> 2. As $K$ increases, how do you expect these two errors to change?

In [ ]:
# The test curve is shown here only to illustrate generalization.
# We should not use it to select K in practice.
plt.plot(K_list, err_test, '-', label='unseen')
plt.plot(K_list, err_train, '-', label='train')
plt.legend()
plt.xlabel('K (number of neighbours)')
plt.ylabel('mean squared error')
plt.show()

| Value of $K$ | Training error | Unseen-data error | Model behavior |
|---|---:|---:|---|
| Small $K$ | Very low | High | **Overfitting** |
| Moderate $K$ | Slightly higher | Lowest | **Best generalization** |
| Very large $K$ | High | High | **Underfitting** |

> **Pause the video and think about this question:**
>
> If we select the best $K$ using training error, will it also minimize the test error?


We care about generalization because we do not want a model that underfits or overfits. We want to find a model that performs well on unseen data—in other words, a model that generalizes well.

## 3. How Do We Select a Model That Generalizes Well?

### 3.1 Why Do We Need a Validation Set?

- Why Can't We Use the Training Set?
    - Our goal is good performance on unseen data.
    - The lowest training error does not imply the lowest generalization error.

- Why Can't We Use the Test Set?
    - **Test leakage problem:** Selecting a model based on test error makes the test set part of the model-selection process.
    - The test set should be reserved for final evaluation.

- The Role of Each Dataset
    - **Training set:** Train the model.
    - **Validation set:** Select the model and hyperparameters.
    - **Test set:** Final evaluation.

![Training, validation, and test data split](assets/data-split.svg)

### 3.2 How Do We Select a Hyperparameter Using a Validation Set?

First, split the data into training, validation, and test sets.

1. Loop over candidate values of the hyperparameter.
2. Train each model using only the training set.
3. Compute the validation error.
4. Select the hyperparameter with the lowest validation error.
5. Combine the training and validation sets.
6. Retrain the model using the selected $K$.
7. Evaluate the final model once on the test set.

In [ ]:
# Split the data into train (80%), validation (10%), and test (10%) sets
n_test = num_instances // 10
n_valid = num_instances // 10

inds = np.random.permutation(num_instances)

x_test, y_test = x[inds[:n_test]], y[inds[:n_test]]
x_valid, y_valid = x[inds[n_test:n_test + n_valid]], y[inds[n_test:n_test + n_valid]]
x_train, y_train = x[inds[n_test + n_valid:]], y[inds[n_test + n_valid:]]

# Select K using validation error only
K_list = range(1, 30)
err_valid = []

for K in K_list:
    model = neighbors.KNeighborsRegressor(n_neighbors=K)
    model.fit(x_train, y_train)
    err_valid.append(loss(y_valid, model.predict(x_valid)))

plt.plot(K_list, err_valid, label='validation')
plt.legend()
plt.xlabel('K (number of neighbours)')
plt.ylabel('mean squared error')
plt.show()

best_index = np.argmin(err_valid)
best_K = K_list[best_index]
print(f'Best K: {best_K}')
print(f'Validation error: {err_valid[best_index]:.4f}')

# Retrain on all non-test data
x_train_final = np.concatenate([x_train, x_valid], axis=0)
y_train_final = np.concatenate([y_train, y_valid], axis=0)

final_model = neighbors.KNeighborsRegressor(n_neighbors=best_K)
final_model.fit(x_train_final, y_train_final)

# Evaluate on the test set once, after model selection
test_error = loss(y_test, final_model.predict(x_test))
print(f'Final test error: {test_error:.4f}')

### 3.3 Why Do We Need Cross-Validation?

- A single validation split can be sensitive to the particular random split.
- Repeating training and validation across multiple folds gives us a more reliable estimate of validation performance.
- Cross-validation reduces our dependence on one particular validation split.

### 3.4 How Do We Perform Cross-Validation?

![Five-fold cross-validation with test set](assets/five-fold-cross-validation.svg)

In [ ]:
def cross_validate(n, n_folds=5):
    """Yield training and validation indices for K-fold cross-validation."""
    
    n_val = n // n_folds
    
    for f in range(n_folds):
        val_inds = list(range(f * n_val, (f + 1) * n_val))
        tr_inds = list(range(0, f * n_val))
        tr_inds += list(range((f + 1) * n_val, n))
        yield tr_inds, val_inds

In [ ]:
# Set aside the test data before cross-validation
inds = np.random.permutation(num_instances)

x_test, y_test = x[inds[:n_test]], y[inds[:n_test]]
x_rest, y_rest = x[inds[n_test:]], y[inds[n_test:]]
n_rest = len(x_rest)

In [ ]:
# Use cross-validation to select K
num_folds = 5
K_list = range(1, 30)
err_valid = np.zeros((len(K_list), num_folds))

for i, K in enumerate(K_list):
    for f, (tr, val) in enumerate(cross_validate(n_rest, num_folds)):
        model = neighbors.KNeighborsRegressor(n_neighbors=K)
        model.fit(x_rest[tr], y_rest[tr])
        err_valid[i, f] = loss(y_rest[val], model.predict(x_rest[val]))

mean_valid_error = np.mean(err_valid, axis=1)
best_index = np.argmin(mean_valid_error)
best_K = K_list[best_index]

print(f'Best K: {best_K}')
print(f'Mean cross-validation error: {mean_valid_error[best_index]:.4f}')

# Retrain on all non-test data
final_model = neighbors.KNeighborsRegressor(n_neighbors=best_K)
final_model.fit(x_rest, y_rest)

# Evaluate on the test set once
test_error = loss(y_test, final_model.predict(x_test))
print(f'Final test error: {test_error:.4f}')

## 4. How Do We Evaluate the Final Model?

So far, we have used a regression example to study generalization and model selection. Now, let's switch to a classification problem and look at a useful tool for understanding a classifier's errors: the confusion matrix.

A confusion matrix shows how predictions compare with the true labels. Rows represent the true classes, columns represent the predicted classes, diagonal entries are correct predictions, and off-diagonal entries are classification errors.

In [ ]:
from sklearn import tree, model_selection

# Load the Iris classification dataset
x, y = datasets.load_iris(return_X_y=True)

# Split into train (80%) and test (20%) sets
x_train, x_test, y_train, y_test = model_selection.train_test_split(
    x, y, test_size=0.2, random_state=1234
)

# Train a decision tree and predict the test labels
yh_test = tree.DecisionTreeClassifier(random_state=1234).fit(
    x_train, y_train
).predict(x_test)

In [ ]:
def confusion_matrix(y, yh):
    """
    Compute a multi-class confusion matrix.

    Rows represent true classes and columns represent predicted classes.
    Entry [i, j] counts samples with true class i predicted as class j.
    """
    n_classes = np.max(y) + 1
    c_matrix = np.zeros((n_classes, n_classes), dtype=int)

    for c1 in range(n_classes):
        for c2 in range(n_classes):
            c_matrix[c1, c2] = np.sum((y == c1) * (yh == c2))

    return c_matrix


cmat = confusion_matrix(y_test, yh_test)
print(cmat)

For binary classification, the four entries of the confusion matrix are:

- True positive (TP)
- True negative (TN)
- False positive (FP)
- False negative (FN)

Metrics such as accuracy, precision, recall, false-positive rate, and false-discovery rate can be calculated from these values.